# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adityag30/FlyRank-internship-ML/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*



The model output is presented as a ranked content action queue to support editorial decision-making. The recommendations are intended to help prioritize human review and should not be treated as automatic publishing decisions.

| Priority | Recommended Action | Reason Code | Description |
|----------|--------------------|-------------|-------------|
| 1 | Refresh Immediately | HIGH_IMPRESSIONS_LOW_CTR | The page receives many impressions but has a low click-through rate, suggesting that titles, metadata, or content relevance should be reviewed. |
| 2 | Review Content Quality | LOW_ENGAGEMENT | Users visit the page but engagement is relatively low, indicating that the content quality or user experience may require improvement. |
| 3 | Monitor Performance | AVERAGE_PERFORMANCE | Current metrics do not suggest an immediate refresh, but performance should continue to be monitored. |
| 4 | No Action | HEALTHY_PAGE | The page shows healthy performance based on the available metrics and does not require immediate intervention. |

These recommendations are intended as **decision-support** for content teams. Final publishing or editing decisions should always involve human review.

In [2]:
from google.colab import userdata
import duckdb
import pandas as pd

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

query = f"""
SELECT
    content_hash_id,
    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_clicks) AS gsc_clicks,
    AVG(gsc_avg_position) AS gsc_avg_position,
    SUM(ga4_sessions) AS ga4_sessions,
    SUM(ga4_engaged_sessions) AS ga4_engaged_sessions
FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
WHERE month='2026-03'
GROUP BY content_hash_id
"""

df = con.sql(query).df()

df["ctr"] = df["gsc_clicks"] / df["gsc_impressions"].replace(0, 1)
df["engagement_rate"] = (
    df["ga4_engaged_sessions"] /
    df["ga4_sessions"].replace(0, 1)
)

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "ctr",
    "engagement_rate"
]

X = df[feature_cols]

y = (
    (df["gsc_impressions"] >= 10000) &
    (df["ctr"] < 0.03)
).astype(int)

print(df.shape)
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(331437, 8)


,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,ctr,engagement_rate
0,content_b7e512995f79d5a6,1140.0,2.0,4.394234,0.0,0.0,0.001754,0.0
1,content_05597932fe4da067,57.0,0.0,2.714744,0.0,0.0,0.000000,0.0
2,content_905aa32a0230694e,149.0,0.0,6.481453,4.0,0.0,0.000000,0.0
3,content_05434271b257bb68,1421.0,6.0,6.320337,9.0,0.0,0.004222,0.0
4,content_d056587ff7faca0c,2770.0,16.0,4.459107,3.0,0.0,0.005776,0.0


In [3]:
# This cell is for CODE (numbers, a query, a check).
import pandas as pd

queue = df.copy()

def assign_action(row):
    if row["gsc_impressions"] >= 10000 and row["ctr"] < 0.03:
        return pd.Series(["Refresh Immediately", "HIGH_IMPRESSIONS_LOW_CTR", 1])

    elif row["engagement_rate"] < 0.50:
        return pd.Series(["Review Content Quality", "LOW_ENGAGEMENT", 2])

    elif row["ctr"] < 0.05:
        return pd.Series(["Monitor Performance", "AVERAGE_PERFORMANCE", 3])

    else:
        return pd.Series(["No Action", "HEALTHY_PAGE", 4])

queue[["Recommended_Action", "Reason_Code", "Priority"]] = queue.apply(
    assign_action,
    axis=1
)

queue = queue.sort_values(
    by=["Priority", "gsc_impressions"],
    ascending=[True, False]
)

display(
    queue[
        [
            "content_hash_id",
            "Priority",
            "Recommended_Action",
            "Reason_Code",
            "gsc_impressions",
            "ctr",
            "engagement_rate",
        ]
    ].head(20)
)

,content_hash_id,Priority,Recommended_Action,Reason_Code,gsc_impressions,ctr,engagement_rate
92495,content_eadb33b5df496f4a,1,Refresh Immediately,HIGH_IMPRESSIONS_LOW_CTR,617124.0,0.009185,0.082051
93390,content_ec2e0346994fb5a5,1,Refresh Immediately,HIGH_IMPRESSIONS_LOW_CTR,245276.0,0.006034,0.110294
104944,content_e8a52cf3d5988c07,1,Refresh Immediately,HIGH_IMPRESSIONS_LOW_CTR,244931.0,0.002731,0.115600
257894,content_0e03de7680314cd5,1,Refresh Immediately,HIGH_IMPRESSIONS_LOW_CTR,221310.0,0.003253,0.126882
45375,content_44f34c0a90047651,1,Refresh Immediately,HIGH_IMPRESSIONS_LOW_CTR,212404.0,0.000113,0.027027
271323,content_7172a7fad43f0998,1,Refresh Immediately,HIGH_IMPRESSIONS_LOW_CTR,205867.0,0.004187,NaN
59680,content_e7b5dd4dff461ad2,1,Refresh Immediately,HIGH_IMPRESSIONS_LOW_CTR,205045.0,0.011929,NaN
257859,content_8d7d99f109e19aa2,1,Refresh Immediately,HIGH_IMPRESSIONS_LOW_CTR,203497.0,0.001420,0.097561
106457,content_f107e54b10b43725,1,Refresh Immediately,HIGH_IMPRESSIONS_LOW_CTR,195997.0,0.005082,NaN
207072,content_36e53e9c707674fc,1,Refresh Immediately,HIGH_IMPRESSIONS_LOW_CTR,194579.0,0.001244,0.013446


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*



### Intended use

This playbook is intended to help SEO specialists, content editors, and website managers prioritize pages for manual review. The ranked action queue provides decision-support by identifying pages that may benefit from content updates based on observable search and engagement metrics.

### Appropriate use

- Prioritize pages for content review.
- Support editorial planning and resource allocation.
- Identify pages that may require content refreshes or further investigation.
- Monitor content performance over time.

### Limits

- The recommendations do not guarantee that refreshing a page will improve search performance.
- The model was evaluated using a random split because grouped client identifiers were not available in the extracted dataset.
- The target label was derived from observable features, which introduces label leakage and likely inflates the reported accuracy.
- The model has not been validated on future time periods or unseen clients.
- The recommendations should be interpreted as directional guidance rather than production-ready automation.

The playbook is designed to support human decision-making and should not replace editorial judgment.

In [4]:
# This cell is for CODE (numbers, a query, a check).
summary = {
    "Pages Evaluated": len(df),
    "Features Used": len(feature_cols),
    "Model Purpose": "Decision Support",
    "Human Review Required": "Yes"
}

print(summary)

{'Pages Evaluated': 331437, 'Features Used': 7, 'Model Purpose': 'Decision Support', 'Human Review Required': 'Yes'}


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*



## Human review checklist

Before acting on any recommendation, a content reviewer should verify:

- The page content is accurate, relevant, and up to date.
- The recommendation aligns with current business goals and SEO strategy.
- The page is not affected by seasonal trends or temporary events.
- Search intent still matches the page content.
- Technical SEO issues (broken links, indexing, page speed, metadata) have been checked.
- Recent manual updates have not already addressed the issue.
- The recommendation is supported by additional performance data, not just a single metric.

## No-go list (Do not automate)

The following actions should **never** be fully automated based only on model output:

- Publishing or rewriting content without human approval.
- Deleting pages.
- Changing page titles or meta descriptions automatically.
- Redirecting URLs.
- Making business or marketing decisions solely from the model's recommendations.
- Treating the ranked queue as proof that a refresh will improve performance.

The ranked action queue is intended as **decision-support**. Final editorial and publishing decisions must always be made by a human reviewer.

In [5]:
# This cell is for CODE (numbers, a query, a check).
review_checklist = {
    "Human review required": True,
    "Automatic publishing allowed": False,
    "Automatic deletion allowed": False,
    "Automatic redirects allowed": False,
    "Editorial approval required": True
}

print(review_checklist)

{'Human review required': True, 'Automatic publishing allowed': False, 'Automatic deletion allowed': False, 'Automatic redirects allowed': False, 'Editorial approval required': True}


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*


The action playbook should be monitored regularly to ensure that its recommendations remain useful and aligned with current content performance.

### Monitoring triggers

- A noticeable decline in click-through rate (CTR) or engagement across recommended pages.
- A large change in search traffic or impressions after search engine algorithm updates.
- New content types or website sections that were not represented in the original dataset.
- Repeated disagreement between model recommendations and editorial review.
- Significant changes in the distribution of key features, such as impressions, CTR, or engagement rate.

### Retrain triggers

The model should be reviewed or retrained when:

- New historical performance data becomes available.
- The content strategy or SEO objectives change substantially.
- Feature distributions differ noticeably from those used during model development.
- Validation metrics decline on newly collected evaluation data.

Because this project was developed as a research prototype and the target label was derived from observable features, any retrained model should also be re-evaluated for data leakage before deployment.

In [6]:
# This cell is for CODE (numbers, a query, a check).
monitoring_summary = {
    "Records Evaluated": len(df),
    "Monitor Features": [
        "gsc_impressions",
        "ctr",
        "engagement_rate"
    ],
    "Human Review Required": True,
    "Retraining Recommended": "When new data or major performance changes are observed"
}

print(monitoring_summary)

{'Records Evaluated': 331437, 'Monitor Features': ['gsc_impressions', 'ctr', 'engagement_rate'], 'Human Review Required': True, 'Retraining Recommended': 'When new data or major performance changes are observed'}


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*



The ranked action queue is exported to **work/outputs/** for use in the final research paper. The notebook regenerates this file each time it is executed, so the exported CSV is not intended to be committed to the repository.

A summary figure showing the distribution of recommended actions is also saved. These exported artifacts provide the supporting evidence for the recommendations discussed in the paper.

In [7]:
# This cell is for CODE (numbers, a query, a check).
from pathlib import Path
import matplotlib.pyplot as plt

# Create output directories if they do not exist
outputs_dir = Path("work/outputs")
figures_dir = Path("work/figures")

outputs_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

# Export ranked action queue
queue_path = outputs_dir / "ranked_action_queue.csv"

queue.to_csv(queue_path, index=False)

# Create a simple figure for the paper
action_counts = queue["Recommended_Action"].value_counts()

plt.figure(figsize=(6,4))
action_counts.plot(kind="bar")
plt.title("Recommended Actions")
plt.xlabel("Action")
plt.ylabel("Number of Pages")
plt.tight_layout()

figure_path = figures_dir / "recommended_actions.png"
plt.savefig(figure_path, dpi=300)
plt.close()

print(f"Queue exported to: {queue_path}")
print(f"Figure exported to: {figure_path}")

Queue exported to: work/outputs/ranked_action_queue.csv
Figure exported to: work/figures/recommended_actions.png


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.